# Assignment 2: Customer Fraud Detection – Binary Classification

Dear Students,

Aap sab ko *Customer Fraud Detection Dataset* assign ki ja rahi hai.

### Task:

Is dataset par *Machine Learning Binary Classification* perform karni hai. Aapka goal customer ke available features ki madad se predict karna hai ke customer/record *fraudulent hai ya nahi*.

### Target Column (Prediction):

* is_fraudulent

### Target Classes:

* 0 = Not Fraudulent
* 1 = Fraudulent

Baaki relevant columns ko features ke taur par use karein.

*Important:* customer_id sirf ek identifier hai, isliye ise Machine Learning model mein feature ke taur par use nahi karna.

### Requirements:

1. Dataset ko properly load karein aur *EDA (Exploratory Data Analysis)* perform karein.
2. Dataset mein:

   * Missing values
   * Duplicate values
   * Outliers
   * Target class distribution
     check karein.
3. Features (X) aur target (y) ko properly separate karein.
4. Categorical columns ko suitable *Encoding Technique* ke through convert karein.
5. Date-related column customer_since ko zarurat ke mutabiq suitable format/features mein convert karein.
6. Dataset ko *Training aur Testing data* mein divide karein.
7. Agar target classes imbalanced hon to *SMOTE* ya koi suitable balancing technique apply karein.
8. Suitable *Feature Scaling* technique apply karein, jaise:

   * StandardScaler
   * MinMaxScaler

   Scaling technique khud select karein aur explain karein ke aapne woh technique kyun choose ki.
9. Neeche diye gaye *Classification Algorithms lazmi apply karein*:

   * Decision Tree
   * Random Forest
   * XGBoost

   Iske ilawa aap koi aur suitable classification algorithm bhi apply kar sakte hain.
10. Har model ki performance evaluate karein using:

* Accuracy
* Precision
* Recall
* F1-Score
* Confusion Matrix
* Classification Report

11. Sabhi models ki performance ko compare karein.
12. *Best-performing model identify karein*.
13. Clearly mention karein:

* Kaunsi encoding technique use ki aur kyun.
* Kaunsi scaling technique use ki aur kyun.
* SMOTE/balancing technique use ki ya nahi aur kyun.
* Decision Tree ki accuracy/performance.
* Random Forest ki accuracy/performance.
* XGBoost ki accuracy/performance.
* Kaunsa model best perform kiya.
* Best model ki final performance kya rahi.

### Important:

Aapko sirf models apply nahi karne. Har preprocessing step aur model selection ko *properly explain* karna hai.

Especially ye explain karna zaroori hai ke:

*Data → Preprocessing → Encoding → Train/Test Split → Scaling → SMOTE (if required) → Models → Evaluation → Model Comparison → Best Model*

### Submission Requirements:

* Complete Jupyter Notebook (.ipynb)
* Dataset
* Proper comments in code
* EDA and visualizations
* Data preprocessing
* Encoding
* Outlier handling
* Feature Scaling
* SMOTE / Class Balancing (if required)
* Decision Tree
* Random Forest
* XGBoost
* Model Evaluation
* Model Comparison
* Confusion Matrix
* Classification Report
* Final Prediction Results
* Best Model explanation

In [1]:
import pandas as pd
import numpy as np

# ==============================================================================
# STEP 1: ORIGINAL DATASET AUDIT
# ==============================================================================
# Objective: Load the raw dataset and run a complete, non-destructive audit.
# Rules enforced: No data modification, no preprocessing, no SMOTE, no modeling.
# ==============================================================================

# 1. Load the dataset (Update file_path to match your exact file name/path)
file_path = 'customer_analytics_dataset.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded dataset from: {file_path}\n")

    # 2. Dataset Shape
    n_rows, n_cols = df.shape
    print(f"--- 1. Dataset Shape ---")
    print(f"Total Rows: {n_rows:,}")
    print(f"Total Columns: {n_cols:,}\n")

    # 3. Column Names & Data Types
    print(f"--- 2. Column Names & Data Types ---")
    print(df.dtypes)
    print("\n")

    # 4. First 5 Rows
    print(f"--- 3. First 5 Rows ---")
    display(df.head())
    print("\n")

    # 5. Missing Values Audit
    print(f"--- 4. Missing Values Audit ---")
    missing_counts = df.isnull().sum()
    missing_total = missing_counts.sum()
    print(f"Total Missing Values across entire dataset: {missing_total:,}")
    if missing_total > 0:
        print("Columns with missing values:")
        print(missing_counts[missing_counts > 0])
    print("\n")

    # 6. Duplicate Rows Audit
    print(f"--- 5. Duplicate Rows Audit ---")
    duplicate_count = df.duplicated().sum()
    print(f"Total Duplicate Rows: {duplicate_count:,}\n")

    # 7. Infinite Values Audit (Checking numeric columns for np.inf or -np.inf)
    print(f"--- 6. Infinite Values Audit ---")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_count = np.isinf(df[numeric_cols]).sum().sum()
    print(f"Total Infinite Values across numeric columns: {inf_count:,}\n")

    # 8. Target Column Candidates
    print(f"--- 7. Target Column Candidates ---")
    # Search for common naming conventions for binary/fraud targets
    keywords = ['fraud', 'target', 'class', 'label', 'is_fraud', 'status']
    target_candidates = [col for col in df.columns if any(k in col.lower() for k in keywords)]
    print(f"Identified Target Candidates: {target_candidates}\n")

    # 9. Target Class Analysis (Assuming 'is_fraudulent' or first candidate, inspect if found)
    # If your target has a specific known name, you can adjust the column check below.
    print(f"--- 8. Target Class Distribution Analysis ---")
    target_col = None
    for cand in target_candidates:
        if 'fraud' in cand.lower() or 'target' in cand.lower():
            target_col = cand
            break
    
    if target_col is None and len(target_candidates) > 0:
        target_col = target_candidates[0] # Default to first candidate if explicit match not found

    if target_col and target_col in df.columns:
        print(f"Analyzing Target Column: '{target_col}'")
        unique_vals = df[target_col].unique()
        class_counts = df[target_col].value_counts(dropna=False)
        class_percentages = df[target_col].value_counts(normalize=True, dropna=False) * 100
        
        print(f"Unique Values: {unique_vals}")
        print("\nClass Counts:")
        print(class_counts)
        print("\nClass Percentages (%):")
        print(class_percentages.round(4))
    else:
        print("Warning: Could not automatically pinpoint target column. Please verify manually from column list.")

    # ==============================================================================
    # DIAGNOSTIC SUMMARY REPORT
    # ==============================================================================
    print("\n" + "=" * 60)
    print("DIAGNOSTIC REPORT: STEP 1 COMPLETED")
    print("=" * 60)
    print(f"• Dataset dimensions: {n_rows:,} rows × {n_cols:,} columns")
    print(f"• Missing values present: {'Yes (' + str(missing_total) + ')' if missing_total > 0 else 'No'}")
    print(f"• Duplicate rows present: {'Yes (' + str(duplicate_count) + ')' if duplicate_count > 0 else 'No'}")
    print(f"• Infinite values present: {'Yes (' + str(inf_count) + ')' if inf_count > 0 else 'No'}")
    print(f"• Target column identified: {target_col if target_col else 'Needs Manual Verification'}")
    print("=" * 60)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please provide the correct path.")
except Exception as e:
    print(f"An error occurred during execution: {e}")

Successfully loaded dataset from: customer_analytics_dataset.csv

--- 1. Dataset Shape ---
Total Rows: 5,000
Total Columns: 13

--- 2. Column Names & Data Types ---
customer_id            object
age                     int64
gender                 object
country                object
avg_order_value       float64
total_orders            int64
last_purchase           int64
is_fraudulent           int64
preferred_category     object
email_open_rate       float64
customer_since         object
loyalty_score           int64
churn_risk            float64
dtype: object


--- 3. First 5 Rows ---


,customer_id,age,gender,country,avg_order_value,total_orders,last_purchase,is_fraudulent,preferred_category,email_open_rate,customer_since,loyalty_score,churn_risk
0,CUST_8270,30,Female,Brazil,101.08,8,176,1,Beauty,25.6,2024-06-05,50,0.20
1,CUST_1860,53,Female,USA,90.39,10,88,0,Electronics,12.3,2024-02-19,37,0.34
2,CUST_6390,73,Male,Australia,83.28,6,203,0,Sports,NaN,2024-04-16,65,0.05
3,CUST_6191,30,Other,Japan,109.90,9,346,1,Electronics,42.9,2020-07-08,93,0.19
4,CUST_6734,29,Female,Canada,269.38,16,342,0,Fashion,5.3,2025-04-09,79,0.15




--- 4. Missing Values Audit ---
Total Missing Values across entire dataset: 500
Columns with missing values:
avg_order_value    250
email_open_rate    250
dtype: int64


--- 5. Duplicate Rows Audit ---
Total Duplicate Rows: 0

--- 6. Infinite Values Audit ---
Total Infinite Values across numeric columns: 0

--- 7. Target Column Candidates ---
Identified Target Candidates: ['is_fraudulent']

--- 8. Target Class Distribution Analysis ---
Analyzing Target Column: 'is_fraudulent'
Unique Values: [1 0]

Class Counts:
is_fraudulent
0    4871
1     129
Name: count, dtype: int64

Class Percentages (%):
is_fraudulent
0    97.42
1     2.58
Name: proportion, dtype: float64

DIAGNOSTIC REPORT: STEP 1 COMPLETED
• Dataset dimensions: 5,000 rows × 13 columns
• Missing values present: Yes (500)
• Duplicate rows present: No
• Infinite values present: No
• Target column identified: is_fraudulent


In [2]:
from scipy import stats
from scipy.stats import chi2_contingency

# ==============================================================================
# STEP 2: TARGET AND FEATURE SIGNAL VALIDATION
# ==============================================================================
# Objective: Assess if the original features contain real predictive signal 
# against the target ('is_fraudulent') using statistical tests and comparisons.
# Rules enforced: No SMOTE, no sampling, no modeling, no dataset modifications.
# ==============================================================================

file_path = 'customer_analytics_dataset.csv'

try:
    df = pd.read_csv(file_path)
    
    # 1. Separate Features and Target
    target_col = 'is_fraudulent'
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    print("=" * 60)
    print("1. NUMERICAL FEATURE STATISTICAL COMPARISON & CORRELATION")
    print("=" * 60)
    
    # Identify numerical columns (excluding potential IDs if present, though we audit all numeric here)
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    
    num_results = []
    for col in num_cols:
        # Separate normal and fraud values (dropping NaNs for calculations)
        normal_vals = df[df[target_col] == 0][col].dropna()
        fraud_vals = df[df[target_col] == 1][col].dropna()
        
        mean_normal = normal_vals.mean()
        mean_fraud = fraud_vals.mean()
        median_normal = normal_vals.median()
        median_fraud = fraud_vals.median()
        
        # Mann-Whitney U test (non-parametric test for distribution differences)
        u_stat, p_val = stats.mannwhitneyu(normal_vals, fraud_vals, alternative='two-sided')
        
        # Point-biserial correlation with the target
        valid_idx = df[col].notna()
        corr, corr_p = stats.pointbiserialr(df.loc[valid_idx, col], y[valid_idx])
        
        num_results.append({
            'Feature': col,
            'Mean (Normal)': mean_normal,
            'Mean (Fraud)': mean_fraud,
            'Mean Diff': mean_fraud - mean_normal,
            'Median (Normal)': median_normal,
            'Median (Fraud)': median_fraud,
            'Mann-Whitney p-val': p_val,
            'Correlation': corr
        })
        
    num_summary_df = pd.DataFrame(num_results)
    print(num_summary_df[['Feature', 'Mean Diff', 'Mann-Whitney p-val', 'Correlation']].to_string(index=False))
    print("\n")

    print("=" * 60)
    print("2. CATEGORICAL FEATURE FRAUD-RATE & CHI-SQUARE ANALYSIS")
    print("=" * 60)
    
    # Identify categorical columns (including object types)
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    # Also check low-cardinality integers if any, but object covers standard categoricals here
    
    cat_results = []
    for col in cat_cols:
        # Contingency table for Chi-Square test
        contingency_table = pd.crosstab(df[col], df[target_col])
        chi2, p_val, dof, ex = chi2_contingency(contingency_table)
        
        print(f"\nFeature: '{col}' (Chi-Square p-value: {p_val:.5e})")
        cat_breakdown = df.groupby(col)[target_col].agg(
            total_count='count',
            fraud_count=lambda x: (x == 1).sum(),
            fraud_rate=lambda x: (x == 1).mean() * 100
        ).reset_index()
        print(cat_breakdown.to_string(index=False))
        
        cat_results.append({
            'Feature': col,
            'Chi2 p-val': p_val
        })

    print("\n" + "=" * 60)
    print("3. CONFLICTING LABELS CHECK (Identical feature patterns with different targets)")
    print("=" * 60)
    # Check if exact feature duplicates exist with opposing target values
    feature_cols_only = [c for c in X.columns if c != 'customer_id'] # Exclude unique ID
    duplicates_check = df.duplicated(subset=feature_cols_only, keep=False)
    n_conflicting = 0
    if duplicates_check.sum() > 0:
        dup_groups = df[duplicates_check].groupby(feature_cols_only)[target_col].nunique()
        n_conflicting = (dup_groups > 1).sum()
    print(f"Number of identical feature rows with conflicting target labels: {n_conflicting}")

    print("\n" + "=" * 60)
    print("4. SUMMARY TABLE: STRONGEST POTENTIAL PREDICTORS")
    print("=" * 60)
    # Combine signals to rank potential predictors based on p-values
    if len(num_summary_df) > 0:
        num_ranked = num_summary_df[['Feature', 'Mann-Whitney p-val', 'Correlation']].copy()
        num_ranked['Absolute Correlation'] = num_ranked['Correlation'].abs()
        num_ranked = num_ranked.sort_values(by='Mann-Whitney p-val')
        print(num_ranked[['Feature', 'Mann-Whitney p-val', 'Absolute Correlation']].to_string(index=False))

    # ==============================================================================
    # DIAGNOSTIC CONCLUSION
    # ==============================================================================
    print("\n" + "=" * 60)
    print("DIAGNOSTIC CONCLUSION: STEP 2")
    print("=" * 60)
    
    # Evaluate significance across numeric and categorical features
    sig_num_count = (num_summary_df['Mann-Whitney p-val'] < 0.05).sum() if len(num_summary_df) > 0 else 0
    sig_cat_count = sum(1 for c in cat_results if c['Chi2 p-val'] < 0.05)
    total_features = len(num_cols) + len(cat_cols)
    sig_total = sig_num_count + sig_cat_count
    
    print(f"Statistically significant features (p < 0.05): {sig_total} out of {total_features}")
    
    if sig_total == 0 or (num_summary_df['Mann-Whitney p-val'].min() > 0.05 and all(c['Chi2 p-val'] > 0.05 for c in cat_results)):
        conclusion = "D. Almost no predictive signal"
        explanation = "None of the numerical or categorical features show statistically significant differences (p > 0.05) or meaningful correlations with the target class, indicating random noise or lack of predictive signal in the original attributes."
    elif sig_total < (total_features / 2):
        conclusion = "C. Weak predictive signal"
        explanation = "Only a small fraction of features show statistically significant associations with the target, while most features exhibit negligible distribution differences between normal and fraud classes."
    elif sig_total < total_features:
        conclusion = "B. Moderate predictive signal"
        explanation = "A moderate subset of features display statistically significant p-values and noticeable differences between normal and fraud records, suggesting partial predictability."
    else:
        conclusion = "A. Strong predictive signal"
        explanation = "Most or all features demonstrate highly significant p-values, clear distribution separations, and meaningful correlations with the target variable."

    print(f"\nConclusion: {conclusion}")
    print(f"Reasoning: {explanation}")
    print("=" * 60)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

1. NUMERICAL FEATURE STATISTICAL COMPARISON & CORRELATION
        Feature  Mean Diff  Mann-Whitney p-val  Correlation
            age   0.365613            0.822462     0.003242
avg_order_value  -4.757455            0.674700    -0.010953
   total_orders   0.043900            0.926285     0.002200
  last_purchase  -5.112068            0.589407    -0.007725
email_open_rate   3.181674            0.244933     0.017090
  loyalty_score   0.691622            0.783977     0.003803
     churn_risk  -0.003568            0.885833    -0.003543


2. CATEGORICAL FEATURE FRAUD-RATE & CHI-SQUARE ANALYSIS

Feature: 'customer_id' (Chi-Square p-value: 4.89045e-01)
customer_id  total_count  fraud_count  fraud_rate
  CUST_1001            2            0    0.000000
  CUST_1002            1            1  100.000000
  CUST_1004            1            0    0.000000
  CUST_1005            1            0    0.000000
  CUST_1006            1            0    0.000000
  CUST_1007            1            0    0.000

In [3]:
import pandas as pd
import numpy as np

# ==============================================================================
# STEP 3: LEAKAGE & FEATURE USAGE AUDIT
# ==============================================================================
# Objective: Audit all 13 columns for leakage, IDs, future info, missing values, 
# and structural suitability before any preprocessing or modeling.
# Rules enforced: No model training, no SMOTE, no data deletion, no feature engineering.
# ==============================================================================

file_path = 'customer_analytics_dataset.csv'

try:
    df = pd.read_csv(file_path)
    
    # Audit Breakdown Table Construction
    audit_data = [
        {"Feature": "customer_id", "Type": "Identifier", "Keep/Drop": "Drop", "Reason": "Unique identifier; risks memorization/overfitting."},
        {"Feature": "age", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Valid demographic predictor; no leakage."},
        {"Feature": "gender", "Type": "Categorical", "Keep/Drop": "Keep", "Reason": "Demographic attribute; safe predictor."},
        {"Feature": "country", "Type": "Categorical", "Keep/Drop": "Keep", "Reason": "Geographic attribute; safe predictor."},
        {"Feature": "avg_order_value", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Transactional history metric; needs imputation (250 missing)."},
        {"Feature": "total_orders", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Behavioral history metric; safe predictor."},
        {"Feature": "last_purchase", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Recency metric; safe predictor."},
        {"Feature": "is_fraudulent", "Type": "Target", "Keep/Drop": "Drop (Target)", "Reason": "Target variable (label), must be separated from features."},
        {"Feature": "preferred_category", "Type": "Categorical", "Keep/Drop": "Keep", "Reason": "Preference metric; safe predictor."},
        {"Feature": "email_open_rate", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Engagement metric; needs imputation (250 missing)."},
        {"Feature": "customer_since", "Type": "Date/Time", "Keep/Drop": "Transform", "Reason": "Date string; convert to tenure/age in days before modeling."},
        {"Feature": "loyalty_score", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Account metric; safe predictor."},
        {"Feature": "churn_risk", "Type": "Numerical", "Keep/Drop": "Keep", "Reason": "Behavioral risk score; safe predictor."}
    ]
    
    audit_df = pd.DataFrame(audit_data)
    
    print("=" * 80)
    print("STEP 3 AUDIT TABLE: FEATURE USAGE & LEAKAGE ASSESSMENT")
    print("=" * 80)
    print(audit_df.to_markdown(index=False))
    print("\n")
    
    # Print Required Summary Section
    print("=== STEP 3 RESULT ===")
    print("Safe features: ['age', 'gender', 'country', 'avg_order_value', 'total_orders', 'last_purchase', 'preferred_category', 'email_open_rate', 'loyalty_score', 'churn_risk']")
    print("Drop features: ['customer_id', 'is_fraudulent']")
    print("Potential date features: ['customer_since']")
    print("Missing-value features: ['avg_order_value', 'email_open_rate']")
    print("Leakage detected: NO")
    print("=" * 80)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

STEP 3 AUDIT TABLE: FEATURE USAGE & LEAKAGE ASSESSMENT
| Feature            | Type        | Keep/Drop     | Reason                                                        |
|:-------------------|:------------|:--------------|:--------------------------------------------------------------|
| customer_id        | Identifier  | Drop          | Unique identifier; risks memorization/overfitting.            |
| age                | Numerical   | Keep          | Valid demographic predictor; no leakage.                      |
| gender             | Categorical | Keep          | Demographic attribute; safe predictor.                        |
| country            | Categorical | Keep          | Geographic attribute; safe predictor.                         |
| avg_order_value    | Numerical   | Keep          | Transactional history metric; needs imputation (250 missing). |
| total_orders       | Numerical   | Keep          | Behavioral history metric; safe predictor.                    |
| last_pu

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ==============================================================================
# STEP 4: LEAKAGE-SAFE PREPROCESSING AND TRAIN/VALIDATION/TEST SPLIT
# ==============================================================================

# Load original dataset
file_path = 'customer_analytics_dataset.csv'
df = pd.read_csv(file_path)

# 1. Date feature transformation
df['customer_since'] = pd.to_datetime(df['customer_since'], errors='coerce')
latest_date = df['customer_since'].max()
df['account_tenure_days'] = (latest_date - df['customer_since']).dt.days
df = df.drop(columns=['customer_since'])

# 2 & 3. Separate Features and Target
y = df['is_fraudulent'].copy()
X = df.drop(columns=['customer_id', 'is_fraudulent'])

# Identify feature types
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# 4 & 5. Stratified 70% Train, 15% Validation, 15% Test Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 6. Preprocessing Pipeline Construction
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# 7. Fit ONLY on X_train, Transform Validation and Test
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Calculate metrics for final report
train_fraud_count = int(y_train.sum())
val_fraud_count = int(y_val.sum())
test_fraud_count = int(y_test.sum())

train_fraud_pct = (train_fraud_count / len(y_train)) * 100
val_fraud_pct = (val_fraud_count / len(y_val)) * 100
test_fraud_pct = (test_fraud_count / len(y_test)) * 100

total_processed_features = X_train_processed.shape[1]
remaining_missing_vals = np.isnan(X_train_processed).sum() + np.isnan(X_val_processed).sum() + np.isnan(X_test_processed).sum()

# 8. Required Output Format
print("=== STEP 4 RESULT ===")
print(f"Train shape: {X_train_processed.shape}")
print(f"Validation shape: {X_val_processed.shape}")
print(f"Test shape: {X_test_processed.shape}")
print(f"Train fraud: {train_fraud_count} ({train_fraud_pct:.2f}%)")
print(f"Validation fraud: {val_fraud_count} ({val_fraud_pct:.2f}%)")
print(f"Test fraud: {test_fraud_count} ({test_fraud_pct:.2f}%)")
print(f"Numerical features: {num_features}")
print(f"Categorical features: {cat_features}")
print(f"Processed feature count: {total_processed_features}")
print(f"Missing values after preprocessing: {remaining_missing_vals}")
print("Preprocessing leakage: NO")

=== STEP 4 RESULT ===
Train shape: (3500, 26)
Validation shape: (750, 26)
Test shape: (750, 26)
Train fraud: 90 (2.57%)
Validation fraud: 20 (2.67%)
Test fraud: 19 (2.53%)
Numerical features: ['age', 'avg_order_value', 'total_orders', 'last_purchase', 'email_open_rate', 'loyalty_score', 'churn_risk', 'account_tenure_days']
Categorical features: ['gender', 'country', 'preferred_category']
Processed feature count: 26
Missing values after preprocessing: 0
Preprocessing leakage: NO


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix
)

# Initialize and train the clean baseline model (No SMOTE, No class weights)
baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(X_train_processed, y_train)

# Predict on validation data using default threshold 0.50
y_val_pred = baseline_model.predict(X_val_processed)
y_val_proba = baseline_model.predict_proba(X_val_processed)[:, 1]

# Calculate required metrics
acc = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred, zero_division=0)
rec = recall_score(y_val, y_val_pred, zero_division=0)
f1 = f1_score(y_val, y_val_pred, zero_division=0)
pr_auc = average_precision_score(y_val, y_val_proba)
roc_auc = roc_auc_score(y_val, y_val_proba)

cm = confusion_matrix(y_val, y_val_pred)
tn, fp, fn, tp = cm.ravel()

pred_normal = int((y_val_pred == 0).sum())
pred_fraud = int((y_val_pred == 1).sum())

# Output in the requested strict format
print("=== STEP 5 BASELINE ===")
print("Model: Logistic Regression")
print("Balancing: None")
print("Threshold: 0.50")
print()
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1: {f1:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print()
print("Confusion Matrix:")
print(f"[[{tn} {fp}]")
print(f" [{fn} {tp}]]")
print()
print(f"Predicted Normal: {pred_normal}")
print(f"Predicted Fraud: {pred_fraud}")
print()
print("Test set used: NO")

=== STEP 5 BASELINE ===
Model: Logistic Regression
Balancing: None
Threshold: 0.50

Accuracy: 0.9733
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
PR-AUC: 0.0311
ROC-AUC: 0.5497

Confusion Matrix:
[[730 0]
 [20 0]]

Predicted Normal: 750
Predicted Fraud: 0

Test set used: NO


In [6]:
from imblearn.over_sampling import SMOTE

# 1. Train Logistic Regression with class_weight='balanced'
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42)
lr_balanced.fit(X_train_processed, y_train)

y_val_pred_bal = lr_balanced.predict(X_val_processed)
y_val_proba_bal = lr_balanced.predict_proba(X_val_processed)[:, 1]

# 2. Train Logistic Regression using SMOTE (applied strictly on X_train_processed to avoid leakage)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

lr_smote = LogisticRegression(random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)

y_val_pred_smote = lr_smote.predict(X_val_processed)
y_val_proba_smote = lr_smote.predict_proba(X_val_processed)[:, 1]

# 3. Evaluate both models on validation set
metrics_bal = {
    'Method': 'Class Weight (Balanced)',
    'Accuracy': accuracy_score(y_val, y_val_pred_bal),
    'Precision': precision_score(y_val, y_val_pred_bal, zero_division=0),
    'Recall': recall_score(y_val, y_val_pred_bal, zero_division=0),
    'F1': f1_score(y_val, y_val_pred_bal, zero_division=0),
    'PR-AUC': average_precision_score(y_val, y_val_proba_bal),
    'ROC-AUC': roc_auc_score(y_val, y_val_proba_bal)
}

metrics_smote = {
    'Method': 'SMOTE',
    'Accuracy': accuracy_score(y_val, y_val_pred_smote),
    'Precision': precision_score(y_val, y_val_pred_smote, zero_division=0),
    'Recall': recall_score(y_val, y_val_pred_smote, zero_division=0),
    'F1': f1_score(y_val, y_val_pred_smote, zero_division=0),
    'PR-AUC': average_precision_score(y_val, y_val_proba_smote),
    'ROC-AUC': roc_auc_score(y_val, y_val_proba_smote)
}

comparison_df = pd.DataFrame([metrics_bal, metrics_smote])

cm_bal = confusion_matrix(y_val, y_val_pred_bal)
cm_smote = confusion_matrix(y_val, y_val_pred_smote)

print("=== STEP 6 COMPARISON TABLE ===")
print(comparison_df.to_string(index=False))
print("\nConfusion Matrix (Class Weight: Balanced):")
print(cm_bal)
print("\nConfusion Matrix (SMOTE):")
print(cm_smote)
print("\n" + "="*40 + "\n")

# Determine best methods dynamically from results
best_f1_method = comparison_df.loc[comparison_df['F1'].idxmax(), 'Method']
best_recall_method = comparison_df.loc[comparison_df['Recall'].idxmax(), 'Method']
best_pruac_method = comparison_df.loc[comparison_df['PR-AUC'].idxmax(), 'Method']

best_f1_val = comparison_df['F1'].max()
best_recall_val = comparison_df['Recall'].max()
best_precision_val = comparison_df.loc[comparison_df['F1'].idxmax(), 'Precision']
best_pruac_val = comparison_df['PR-AUC'].max()

print("=== STEP 6 MAIN RESULTS ===")
print(f"Best method by F1: {best_f1_method}")
print(f"Best method by Recall: {best_recall_method}")
print(f"Best method by PR-AUC: {best_pruac_method}")
print(f"Validation F1: {best_f1_val:.4f}")
print(f"Validation Recall: {best_recall_val:.4f}")
print(f"Validation Precision: {best_precision_val:.4f}")
print(f"Validation PR-AUC: {best_pruac_val:.4f}")
print("Leakage: NO")
print("Test set used: NO")

=== STEP 6 COMPARISON TABLE ===
                 Method  Accuracy  Precision  Recall       F1   PR-AUC  ROC-AUC
Class Weight (Balanced)     0.540   0.034384     0.6 0.065041 0.030244 0.536849
                  SMOTE     0.536   0.034091     0.6 0.064516 0.030060 0.542603

Confusion Matrix (Class Weight: Balanced):
[[393 337]
 [  8  12]]

Confusion Matrix (SMOTE):
[[390 340]
 [  8  12]]


=== STEP 6 MAIN RESULTS ===
Best method by F1: Class Weight (Balanced)
Best method by Recall: Class Weight (Balanced)
Best method by PR-AUC: Class Weight (Balanced)
Validation F1: 0.0650
Validation Recall: 0.6000
Validation Precision: 0.0344
Validation PR-AUC: 0.0302
Leakage: NO
Test set used: NO


In [7]:
# ==============================================================================
# STEP 7: VALIDATION THRESHOLD OPTIMIZATION (FIXED)
# ==============================================================================

# Define thresholds since it was missing in this scope
thresholds = np.arange(0.05, 0.96, 0.01)

results = []
for th in thresholds:
    y_pred_th = (y_val_proba_bal >= th).astype(int)
    
    acc = accuracy_score(y_val, y_pred_th)
    prec = precision_score(y_val, y_pred_th, zero_division=0)
    rec = recall_score(y_val, y_pred_th, zero_division=0)
    f1_val = f1_score(y_val, y_pred_th, zero_division=0)
    
    results.append({
        'threshold': th,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1_val
    })

results_df = pd.DataFrame(results)

# Select best F1 row using lowercase 'f1'
best_f1_row = results_df.loc[results_df['f1'].idxmax()]

# Precision constraint check
valid_prec_df = results_df[results_df['precision'] >= 0.10]

# Predictions and Confusion Matrix at best-F1 threshold
y_pred_best_f1 = (y_val_proba_bal >= best_f1_row['threshold']).astype(int)
cm_best_f1 = confusion_matrix(y_val, y_pred_best_f1)

# Metrics calculation
final_acc = accuracy_score(y_val, y_pred_best_f1)
final_pr_auc = average_precision_score(y_val, y_val_proba_bal)
final_roc_auc = roc_auc_score(y_val, y_val_proba_bal)

# Strict requested output format
print("=== STEP 7 MAIN RESULTS ===")
print(f"Best threshold: {best_f1_row['threshold']:.2f}")
print(f"Precision: {best_f1_row['precision']:.4f}")
print(f"Recall: {best_f1_row['recall']:.4f}")
print(f"F1: {best_f1_row['f1']:.4f}")
print(f"Accuracy: {final_acc:.4f}")
print(f"PR-AUC: {final_pr_auc:.4f}")
print(f"ROC-AUC: {final_roc_auc:.4f}")
print("Confusion Matrix:")
print(cm_best_f1)
print("Test set used: NO")
print("Leakage: NO")

=== STEP 7 MAIN RESULTS ===
Best threshold: 0.55
Precision: 0.0364
Recall: 0.4500
F1: 0.0674
Accuracy: 0.6680
PR-AUC: 0.0302
ROC-AUC: 0.5368
Confusion Matrix:
[[492 238]
 [ 11   9]]
Test set used: NO
Leakage: NO


In [8]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

# 1. Train Random Forest with class_weight='balanced'
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train_processed, y_train)
y_pred_rf = rf.predict(X_val_processed)
y_proba_rf = rf.predict_proba(X_val_processed)[:, 1]

# 2. Train HistGradientBoosting (handles class imbalance via class_weight not directly supported via simple parameter, but built well for tabular data)
hgb = HistGradientBoostingClassifier(random_state=42)
hgb.fit(X_train_processed, y_train)
y_pred_hgb = hgb.predict(X_val_processed)
y_proba_hgb = hgb.predict_proba(X_val_processed)[:, 1]

# 3. Train SVM with class_weight='balanced' and probability=True
svm = SVC(class_weight='balanced', probability=True, random_state=42)
svm.fit(X_train_processed, y_train)
y_pred_svm = svm.predict(X_val_processed)
y_proba_svm = svm.predict_proba(X_val_processed)[:, 1]

# Evaluation function helper
def get_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'PR-AUC': average_precision_score(y_true, y_proba),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

m_rf = get_metrics(y_val, y_pred_rf, y_proba_rf)
m_hgb = get_metrics(y_val, y_pred_hgb, y_proba_hgb)
m_svm = get_metrics(y_val, y_pred_svm, y_proba_svm)

comparison_df = pd.DataFrame([
    {'Model': 'Random Forest', **m_rf},
    {'Model': 'HistGradientBoosting', **m_hgb},
    {'Model': 'SVM', **m_svm}
])

print("=== STEP 8 COMPARISON TABLE ===")
print(comparison_df.to_string(index=False))
print("\nConfusion Matrix (Random Forest):")
print(confusion_matrix(y_val, y_pred_rf))
print("\nConfusion Matrix (HistGradientBoosting):")
print(confusion_matrix(y_val, y_pred_hgb))
print("\nConfusion Matrix (SVM):")
print(confusion_matrix(y_val, y_pred_svm))
print("\n" + "="*40 + "\n")

# Determine best models dynamically
best_f1_model = comparison_df.loc[comparison_df['F1'].idxmax(), 'Model']
best_pruac_model = comparison_df.loc[comparison_df['PR-AUC'].idxmax(), 'Model']
best_roauc_model = comparison_df.loc[comparison_df['ROC-AUC'].idxmax(), 'Model']

print("=== STEP 8 MAIN RESULTS ===")
print(f"Best model by F1: {best_f1_model}")
print(f"Best model by PR-AUC: {best_pruac_model}")
print(f"Best model by ROC-AUC: {best_roauc_model}")
print(f"Best validation F1: {comparison_df['F1'].max():.4f}")
print(f"Best validation Precision: {comparison_df['Precision'].max():.4f}")
print(f"Best validation Recall: {comparison_df['Recall'].max():.4f}")
print(f"Best validation PR-AUC: {comparison_df['PR-AUC'].max():.4f}")
print(f"Best validation ROC-AUC: {comparison_df['ROC-AUC'].max():.4f}")
print("Test set used: NO")
print("Leakage: NO")

=== STEP 8 COMPARISON TABLE ===
               Model  Accuracy  Precision  Recall     F1   PR-AUC  ROC-AUC
       Random Forest  0.973333   0.000000     0.0 0.0000 0.104886 0.604623
HistGradientBoosting  0.973333   0.000000     0.0 0.0000 0.059806 0.607877
                 SVM  0.920000   0.045455     0.1 0.0625 0.024207 0.441233

Confusion Matrix (Random Forest):
[[730   0]
 [ 20   0]]

Confusion Matrix (HistGradientBoosting):
[[730   0]
 [ 20   0]]

Confusion Matrix (SVM):
[[688  42]
 [ 18   2]]


=== STEP 8 MAIN RESULTS ===
Best model by F1: SVM
Best model by PR-AUC: Random Forest
Best model by ROC-AUC: HistGradientBoosting
Best validation F1: 0.0625
Best validation Precision: 0.0455
Best validation Recall: 0.1000
Best validation PR-AUC: 0.1049
Best validation ROC-AUC: 0.6079
Test set used: NO
Leakage: NO


In [9]:
# ==============================================================================
# STEP 9: THRESHOLD OPTIMIZATION FOR RANDOM FOREST & HISTGRADIENTBOOSTING
# ==============================================================================

# Extract validation probabilities for Random Forest and HistGradientBoosting
y_proba_rf_val = rf.predict_proba(X_val_processed)[:, 1]
y_proba_hgb_val = hgb.predict_proba(X_val_processed)[:, 1]

thresholds = np.arange(0.01, 1.00, 0.01)

def optimize_thresholds(y_true, y_proba, model_name):
    res = []
    for th in thresholds:
        y_pred_th = (y_proba >= th).astype(int)
        acc = accuracy_score(y_true, y_pred_th)
        prec = precision_score(y_true, y_pred_th, zero_division=0)
        rec = recall_score(y_true, y_pred_th, zero_division=0)
        f1_val = f1_score(y_true, y_pred_th, zero_division=0)
        res.append({'threshold': th, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1_val})
    
    df_res = pd.DataFrame(res)
    
    # A. Maximum F1 threshold
    best_f1 = df_res.loc[df_res['f1'].idxmax()]
    
    # B & C. Constraints (Precision >= 10% and >= 20%)
    prec_10 = df_res[df_res['precision'] >= 0.10]
    prec_20 = df_res[df_res['precision'] >= 0.20]
    
    best_f1_10 = prec_10.loc[prec_10['f1'].idxmax()] if not prec_10.empty else best_f1
    best_f1_20 = prec_20.loc[prec_20['f1'].idxmax()] if not prec_20.empty else best_f1
    
    return {
        'Model': model_name,
        'Best F1 Row': best_f1,
        'Best F1 >= 10%': best_f1_10,
        'Best F1 >= 20%': best_f1_20,
        'DataFrame': df_res
    }

rf_opt = optimize_thresholds(y_val, y_proba_rf_val, 'Random Forest')
hgb_opt = optimize_thresholds(y_val, y_proba_hgb_val, 'HistGradientBoosting')

# Construct summary table rows based on Absolute Maximum F1 (Requirement A)
table_data = [
    {
        'Model': rf_opt['Model'],
        'Best Threshold': rf_opt['Best F1 Row']['threshold'],
        'Precision': rf_opt['Best F1 Row']['precision'],
        'Recall': rf_opt['Best F1 Row']['recall'],
        'F1': rf_opt['Best F1 Row']['f1'],
        'Accuracy': rf_opt['Best F1 Row']['accuracy']
    },
    {
        'Model': hgb_opt['Model'],
        'Best Threshold': hgb_opt['Best F1 Row']['threshold'],
        'Precision': hgb_opt['Best F1 Row']['precision'],
        'Recall': hgb_opt['Best F1 Row']['recall'],
        'F1': hgb_opt['Best F1 Row']['f1'],
        'Accuracy': hgb_opt['Best F1 Row']['accuracy']
    }
]

step9_table = pd.DataFrame(table_data)

# Determine overall best model by F1
overall_best = step9_table.loc[step9_table['F1'].idxmax()]
best_model_name = overall_best['Model']
best_row_data = rf_opt['Best F1 Row'] if best_model_name == 'Random Forest' else hgb_opt['Best F1 Row']
best_proba = y_proba_rf_val if best_model_name == 'Random Forest' else y_proba_hgb_val

pr_auc_val = average_precision_score(y_val, best_proba)
roc_auc_val = roc_auc_score(y_val, best_proba)

print("=== STEP 9 MAIN RESULTS ===")
print(step9_table.to_string(index=False))
print()
print(f"Best model by F1: {best_model_name}")
print(f"Best threshold: {best_row_data['threshold']:.2f}")
print(f"Precision: {best_row_data['precision']:.4f}")
print(f"Recall: {best_row_data['recall']:.4f}")
print(f"F1: {best_row_data['f1']:.4f}")
print(f"Accuracy: {best_row_data['accuracy']:.4f}")
print(f"PR-AUC: {pr_auc_val:.4f}")
print(f"ROC-AUC: {roc_auc_val:.4f}")
print("Test set used: NO")
print("Leakage: NO")

=== STEP 9 MAIN RESULTS ===
               Model  Best Threshold  Precision  Recall       F1  Accuracy
       Random Forest            0.09       0.15    0.15 0.150000  0.954667
HistGradientBoosting            0.10       0.50    0.05 0.090909  0.973333

Best model by F1: Random Forest
Best threshold: 0.09
Precision: 0.1500
Recall: 0.1500
F1: 0.1500
Accuracy: 0.9547
PR-AUC: 0.1049
ROC-AUC: 0.6046
Test set used: NO
Leakage: NO


In [10]:
# ==============================================================================
# STEP 10: FINAL TEST EVALUATION (USING ORIGINAL STEP 9 RANDOM FOREST)
# ==============================================================================

# Use the original test set processed through the original preprocessor pipeline
X_test_proc = preprocessor.transform(X_test)

# Predict probabilities using the original locked Random Forest model (rf) from Step 8/9
y_test_proba = rf.predict_proba(X_test_proc)[:, 1]

# Apply the locked optimal threshold from Step 9 (0.09)
locked_threshold = 0.09
y_test_pred = (y_test_proba >= locked_threshold).astype(int)

# Calculate metrics on the untouched test set
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, zero_division=0)
test_rec = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_pr_auc = average_precision_score(y_test, y_test_proba)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

test_cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = test_cm.ravel()

# Required Output Format
print("=== STEP 10 MAIN RESULTS ===")
print(f"Model: Random Forest")
print(f"Threshold: {locked_threshold:.2f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall: {test_rec:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test PR-AUC: {test_pr_auc:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")
print()
print("Confusion Matrix:")
print(f"[[{tn} {fp}]")
print(f" [{fn} {tp}]]")
print()
print("Test set used: YES")
print("Leakage: NO")

=== STEP 10 MAIN RESULTS ===
Model: Random Forest
Threshold: 0.09
Test Accuracy: 0.9547
Test Precision: 0.0000
Test Recall: 0.0000
Test F1: 0.0000
Test PR-AUC: 0.0235
Test ROC-AUC: 0.4511

Confusion Matrix:
[[716 15]
 [19 0]]

Test set used: YES
Leakage: NO


In [11]:
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# ==============================================================================
# STEP 11: ADDITIONAL REQUIRED MODELS (DECISION TREE & XGBOOST)
# ==============================================================================

# 1. Train Decision Tree with class_weight='balanced'
dt_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt_model.fit(X_train_processed, y_train)

y_pred_dt = dt_model.predict(X_val_processed)
y_proba_dt = dt_model.predict_proba(X_val_processed)[:, 1]

# 2. Train XGBoost (handling 2.58% class imbalance via scale_pos_weight)
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_model = XGBClassifier(scale_pos_weight=scale_pos, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_processed, y_train)

y_pred_xgb = xgb_model.predict(X_val_processed)
y_proba_xgb = xgb_model.predict_proba(X_val_processed)[:, 1]

# 3. Evaluate both models on validation set
dt_metrics = {
    'Model': 'Decision Tree',
    'Accuracy': accuracy_score(y_val, y_pred_dt),
    'Precision': precision_score(y_val, y_pred_dt, zero_division=0),
    'Recall': recall_score(y_val, y_pred_dt, zero_division=0),
    'F1': f1_score(y_val, y_pred_dt, zero_division=0),
    'PR-AUC': average_precision_score(y_val, y_proba_dt),
    'ROC-AUC': roc_auc_score(y_val, y_proba_dt)
}

xgb_metrics = {
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_val, y_pred_xgb),
    'Precision': precision_score(y_val, y_pred_xgb, zero_division=0),
    'Recall': recall_score(y_val, y_pred_xgb, zero_division=0),
    'F1': f1_score(y_val, y_pred_xgb, zero_division=0),
    'PR-AUC': average_precision_score(y_val, y_proba_xgb),
    'ROC-AUC': roc_auc_score(y_val, y_proba_xgb)
}

additional_df = pd.DataFrame([dt_metrics, xgb_metrics])

print("=== STEP 11 ADDITIONAL MODELS RESULTS ===")
print(additional_df.to_string(index=False))
print("\n--- Decision Tree Classification Report ---")
print(classification_report(y_val, y_pred_dt, zero_division=0))
print("--- XGBoost Classification Report ---")
print(classification_report(y_val, y_pred_xgb, zero_division=0))
print("\nTest set used: NO")
print("Leakage: NO")

=== STEP 11 ADDITIONAL MODELS RESULTS ===
        Model  Accuracy  Precision  Recall       F1   PR-AUC  ROC-AUC
Decision Tree     0.960   0.000000    0.00 0.000000 0.026667 0.493151
      XGBoost     0.972   0.333333    0.05 0.086957 0.081892 0.547055

--- Decision Tree Classification Report ---
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       730
           1       0.00      0.00      0.00        20

    accuracy                           0.96       750
   macro avg       0.49      0.49      0.49       750
weighted avg       0.95      0.96      0.95       750

--- XGBoost Classification Report ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       730
           1       0.33      0.05      0.09        20

    accuracy                           0.97       750
   macro avg       0.65      0.52      0.54       750
weighted avg       0.96      0.97      0.96       750


Test set u

In [12]:
# ==============================================================================
# STEP 12: MASTER ASSIGNMENT REPORT DATAFRAME PRESENTATION
# ==============================================================================

# Display the master comparison dataframe cleanly with all requested metrics
master_display_df = master_comparison_df.copy()

# Format float columns for clean presentation
float_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'PR-AUC', 'ROC-AUC']
for col in float_cols:
    master_display_df[col] = master_display_df[col].round(4)


print("=== FINAL ASSIGNMENT MASTER DATAFRAME ===")
display(master_display_df)

NameError: name 'master_comparison_df' is not defined